# 8a. Optimal fingerprinting: calculations

**The question.** Are the observed 1979–2013 trends explained by the forcings, and is each forcing's simulated size right? The main paper attributes the forced changes in the models and checks the models against ERA5. It assumes that each forcing's simulated response has the right size; fingerprinting is the one test of that assumption. It is a separate question, possibly for a separate paper: the record is short (ERA5 from 1979), and the results may well be inconclusive outside the Peninsula and West Antarctica in summer.

**The idea.** A regression of the observations on the models' patterns:

$$
\mathbf{y} = \beta_1 \mathbf{x}_1 + \beta_2 \mathbf{x}_2 + \cdots + \boldsymbol{\varepsilon},
$$

- $\mathbf{y}$: the observed trends, one per region and season, as one vector;
- $\mathbf{x}_i$: the models' forced trend for forcing $i$ in the same entries (its *fingerprint*);
- $\beta_i$: the scaling factor, how much of that pattern the observations hold;
- $\boldsymbol{\varepsilon}$: internal variability, with covariance $\mathbf{C}$.

"Optimal" means the fit is weighted by $\mathbf{C}^{-1}$: entries that vary a lot on their own count less, quiet ones more. Because the trends are per season, the fit can tell forcings apart that look alike in time: greenhouse gases warm every season, ozone mostly the summer.

**Reading β** (with its 90% range, the usual convention of detection and attribution):

| The range of $\beta_i$ | Meaning |
|---|---|
| excludes 0 | detected |
| includes 1 | consistent: the models have the right size |
| above 1 | the models underestimate the response |
| below 1 | the models overestimate it |

A forcing's change is **attributable** where it is detected and consistent *and* the residual test passes (step 9): what the fit leaves over must look like internal variability. Then $\beta_i \mathbf{x}_i$ is its attributable trend.

**The steps** (`extant/fingerprinting.py`, `fp`). This notebook works everything out, with a quick look at each step, and saves the results in `RESULTS_DIR`; notebook 08b opens them and draws the figures (`extant/plots/fingerprinting.py`, `fpp`).

| Step | |
|---|---|
| 0 | the toy example of the planning notes, with a known answer |
| 1 | regional means of ERA5 and of every member, each season, 1979–2013 |
| 2 | their trends, °C per decade |
| 3 | the trends as one vector, region-major |
| 4 | the fingerprints: each model's ensemble mean, then the mean across models, and their effective ensemble sizes |
| 5 | noise samples (members minus their ensemble mean) in two independent halves, and their covariance (Ledoit–Wolf) |
| 6 | prewhitening by $\mathbf{C}_1^{-1/2}$ |
| 7 | the fit: ordinary (OLS) and total least squares (TLS), for one to four signals |
| 8 | the uncertainty, by Monte Carlo |
| 9 | the residual consistency test |
| 10 | what the result says: detected, consistent, attributable trends |
| 11–12 | collinearity, and robustness: each model's own fingerprints, leaving each model out |

**References.** Allen & Tett (1999, *Clim. Dyn.*): the consistency test. Allen & Stott (2003, *Clim. Dyn.*): TLS. Ribes, Planton & Terray (2013, *Clim. Dyn.*): regularised optimal fingerprinting. Hegerl & Zwiers (2011, *WIREs Clim. Change*): a review. Ledoit & Wolf (2004, *J. Multivar. Anal.*): the shrunk covariance.

Names end in their type: `_tree` (DataTree), `_ds` (Dataset), `_da` (DataArray), `_arr` (numpy array), `_df` (DataFrame).

## Setup

In [ ]:
import sys
sys.path[:0] = ['..']   # this repository: the extant package and plotting_modules

import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
import pandas as pd
import xarray as xr

from extant import config, datatree, era5_evaluation as ev, fingerprinting as fp, observed_change as oc
from extant import preprocessing as pp
from extant.logger import setup_logging
from extant.plots import charts, forced_response as fr, methods

logger = setup_logging('extant', logging.INFO)
xr.set_options(display_expand_attrs=False, display_expand_data=False)

#(t): Harmless warnings, silenced the same way in every notebook. zarr's, that parts of the stores (the text
#(t): coordinates, the consolidated metadata) aren't in the Zarr 3 standard yet:
warnings.filterwarnings('ignore', message='.*Zarr (format 3|V3) specification')
#(t): numpy's, about slices with nothing but NaN (land, a model without a member) or nothing at all, too few members for
#(t): a standard deviation, and dividing by zero or by NaN (the log of a probability of 0, say):
warnings.filterwarnings('ignore', message='All-NaN slice encountered', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Mean of empty slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='Degrees of freedom <= 0 for slice', category=RuntimeWarning)
warnings.filterwarnings('ignore', message='(divide by zero|invalid value) encountered', category=RuntimeWarning)
#(t): dask's, about large task graphs, which are large on purpose:
warnings.filterwarnings('ignore', message='.*Sending large graph of size', category=UserWarning)
#(t): Every figure's text at the notebook's sizes and colours (fr.NOTEBOOK), as in every notebook
plt.rcParams.update(fr.NOTEBOOK.rc)

In [ ]:
%matplotlib inline

## Settings

In [ ]:
VARIABLE = 'tas'
var = config.VARIABLES[VARIABLE]

#(t): The years of the trends: ERA5 from 1979, and LESFMIP ends with DJF 2013 (every experiment is cut at 2014)
YEARS = slice(config.ERA5_START_YEAR, 2013)

#(t): The seasons, in the order of the vector's entries within each region
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']

#(t): The fits: ordinary and total least squares; TLS for the final result; the signal set shown step by step
METHODS = ['ols', 'tls']
METHOD = 'tls'
MAIN_SET = '4 signals'

#(t): Monte Carlo trials, for the uncertainty and the residual test, and the range of β reported (5-95%: 90%)
N_MONTE_CARLO = 1000
RANGE = (5, 95)

#(t): Multiplies the noise covariance: above 1 if ERA5 is noisier than the models (notebook 02's noise tests say)
NOISE_INFLATION = 1.0

In [ ]:
#(t): Where notebook 01's seasonal means are, and where this notebook's results go: JASMIN's scratch
SCRATCH_DIR = Path('/work/scratch-pw4/aborow/ExtAnt')
SCRATCH_DIR

## 0. Toy Example

Two regions × two seasons, the observed trends (°C per decade):

| | DJF | JJA |
|---|---|---|
| Peninsula | 0.4 | 0.4 |
| East Antarctica | 0.1 | 0.2 |

Read row by row, $\mathbf{y} = (0.4, 0.4, 0.1, 0.2)$. A greenhouse-gas fingerprint warms everywhere, $\mathbf{x}_{\mathrm{GHG}} = (0.2, 0.4, 0.1, 0.2)$, and an ozone one only the Peninsula in summer, $\mathbf{x}_{\mathrm{O_3}} = (0.2, 0, 0, 0)$. Then $\beta_{\mathrm{GHG}} = \beta_{\mathrm{O_3}} = 1$ rebuilds $\mathbf{y}$ exactly: GHG touches every entry and ozone only the first, which is what lets the fit separate them. Half of the Peninsula's summer warming is from ozone.

In [ ]:
#(t): The observed trends, region by region, and the two fingerprints as columns
toy_y_arr = np.array([0.4, 0.4, 0.1, 0.2])
toy_X_arr = np.column_stack([[0.2, 0.4, 0.1, 0.2], [0.2, 0.0, 0.0, 0.0]])
toy_X_arr

In [ ]:
#(t): The fit, with no noise covariance to weight by (C = I)
toy_beta_arr = fp.fit_ols(toy_y_arr, toy_X_arr, np.eye(4))
toy_beta_arr

In [ ]:
#(t): Each signal's part of every entry, and their sum against the observations
pd.DataFrame({'GHG': toy_beta_arr[0] * toy_X_arr[:, 0], 'O3': toy_beta_arr[1] * toy_X_arr[:, 1],
              'sum': toy_X_arr @ toy_beta_arr, 'observed': toy_y_arr},
             index=['Peninsula DJF', 'Peninsula JJA', 'East DJF', 'East JJA']).round(2)

In [ ]:
#(t): Quick look: each entry's observed trend (black) and the fit's two parts, stacked
fig, ax = plt.subplots(figsize=(7, 3))
entries = ['Peninsula DJF', 'Peninsula JJA', 'East DJF', 'East JJA']
ax.bar(entries, toy_beta_arr[0] * toy_X_arr[:, 0], label='GHG')
ax.bar(entries, toy_beta_arr[1] * toy_X_arr[:, 1], bottom=toy_beta_arr[0] * toy_X_arr[:, 0], label='O3')
ax.plot(entries, toy_y_arr, 'kD', label='observed')
ax.legend();

## Open the Data

Run **one** of the next two sections. Both leave `lesfmip_season_tree` (seasonal means, `/<model>/<experiment>`), `era5_seasonal_da`, `REGIONS` (the regions whose trends are the vector's entries) and `RESULTS_DIR`, where the results are saved.

### Sample data

The 3×3 block of cells of `tests/data`. Its 'regions' are its three rows of cells, so that the vector has more entries (3 × 4) than there are signals.

In [ ]:
#(t): The sample's files, in this repository: <model>_<experiment>_sample.nc, monthly tas in K on (member, time, lat, lon)
SAMPLE_DIR = Path('../tests/data')
sample_files = sorted(SAMPLE_DIR.glob('*_*_sample.nc'))
sample_files

In [ ]:
#(t): The model and experiment of each file, from its name
sample_names = [path.stem.removesuffix('_sample').rsplit('_', 1) for path in sample_files]
sample_names

In [ ]:
#(t): Each file of config.MAIN_EXPERIMENTS opened, as a DataTree /<model>/<experiment>
lesfmip_sample_tree = xr.DataTree.from_dict({f'{model}/{experiment}': xr.open_dataset(path).load()
                                             for path, (model, experiment) in zip(sample_files, sample_names)
                                             if experiment in config.MAIN_EXPERIMENTS})
lesfmip_sample_tree

In [ ]:
#(t): Seasonal means in analysis units, through the same pre-processing as notebook 01
lesfmip_season_tree = pp.seasonal_tree(lesfmip_sample_tree, VARIABLE).compute()
lesfmip_season_tree

In [ ]:
#(t): ERA5's sample file: monthly tas in K from 1979, on (time, lat, lon)
era5_monthly_da = xr.open_dataarray(SAMPLE_DIR / 'era5_sample.nc').load()
era5_monthly_da

In [ ]:
#(t): Its seasonal means in analysis units, on the models' grid (as in notebook 01)
era5_seasonal_da = pp.seasonal_era5(era5_monthly_da, VARIABLE, like=lesfmip_season_tree)
era5_seasonal_da

In [ ]:
#(t): The sample's 'regions': its three rows of cells
REGIONS = {f'{abs(lat):g}°S': {'lat': slice(lat - 1, lat + 1)} for lat in era5_seasonal_da['lat'].values}
REGIONS

In [ ]:
#(t): Where the sample's results are saved
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE / 'sample'
RESULTS_DIR

### Full data

The seasonal means of notebook 01, and the regions of notebook 02 (`observed_change.REGIONS`). Which regions to use is still open: the planning notes suggest starting with a robust subset (the Peninsula and West Antarctica, DJF and MAM) and fixing the regions from the verification (notebook 02), not from what fits best.

In [ ]:
#(t): The two stores notebook 01 saved
seasonal_path = SCRATCH_DIR / 'seasonal' / f'lesfmip_{VARIABLE}.zarr'
era5_path = SCRATCH_DIR / 'seasonal' / f'era5_{VARIABLE}.zarr'
seasonal_path, era5_path

In [ ]:
#(t): The seasonal means, lazily, only the experiments analysed (filter keeps the model nodes above them)
lesfmip_season_tree = xr.open_datatree(seasonal_path, engine='zarr', chunks={})
lesfmip_season_tree = lesfmip_season_tree.filter(lambda node: node.name in config.MAIN_EXPERIMENTS)
lesfmip_season_tree

In [ ]:
#(t): ERA5's seasonal means
era5_seasonal_da = xr.open_zarr(era5_path)[VARIABLE].load()
era5_seasonal_da

In [ ]:
#(t): The regions of notebook 02 (observed_change.REGIONS)
REGIONS = oc.REGIONS
REGIONS

In [ ]:
#(t): Where the full data's results are saved
RESULTS_DIR = SCRATCH_DIR / 'results' / VARIABLE / 'full'
RESULTS_DIR

### What was opened

Whichever section ran: how many members each model ran for each experiment. An empty square is a model and experiment that was never opened.

In [ ]:
#(t): Members per model (rows) and experiment (columns)
member_counts_df = pd.Series({(node.parent.name, node.name): node.sizes['member']
                              for node in lesfmip_season_tree.leaves}).unstack()
member_counts_df = member_counts_df.reindex(columns=config.MAIN_EXPERIMENTS)
member_counts_df

In [ ]:
#(t): The members: each model (rows) and experiment (columns), grey where there are none
fig = plt.figure(figsize=charts.chart_figure_size(n_rows=1, n_cols=1, n_legends=0, n_colorbars=1,
                                                  panel_width=0.9 * member_counts_df.shape[1] + 0.5,
                                                  panel_height=0.4 * member_counts_df.shape[0] + 0.3, left_margin=1.8),
                 layout='constrained')
axes = fig.add_gridspec(1, 1).subplots(squeeze=False)
counts_image = methods.plot_member_counts(member_counts_df, ax=axes[0, 0])
fig.colorbar(counts_image, ax=axes, location='bottom', shrink=0.6, label='Members')
fig.suptitle('LESFMIP members per model and experiment', x=0.01, ha='left');

## 1. Time Series: Regional Means

For every region and season, the area-weighted (cos latitude) mean of each year, for ERA5 and for every member of every model and experiment, over `YEARS`. Every model is on the common 2.5° grid, so no regridding is needed.

In [ ]:
%%time
#(t): Every member's regional means, on (member, region, year, season)
regional_tree = oc.regional_means(lesfmip_season_tree.sel(year=YEARS), REGIONS).compute()
regional_tree

In [ ]:
#(t): ERA5's, on (region, year, season)
era5_regional_da = oc.regional_means(era5_seasonal_da.sel(year=YEARS), REGIONS)
era5_regional_da

In [ ]:
#(t): Quick look: ERA5 (black) and the first model's historical members (grey), the first region
first_model = list(regional_tree.children)[0]
regional_members_da = regional_tree[first_model]['historical'][VARIABLE].isel(region=0)
fig, axes = plt.subplots(1, 4, figsize=(18, 3.2), sharex=True)
for ax, season in zip(axes, SEASONS):
    ax.plot(regional_members_da.year, regional_members_da.sel(season=season).transpose('year', 'member'), color='0.75', lw=0.5)
    ax.plot(era5_regional_da.year, era5_regional_da.isel(region=0).sel(season=season), color='k')
    ax.set_title(season)

## 2. Trends

The least-squares slope of each series, per decade (`ev.trend`, as in notebook 02).

In [ ]:
#(t): Every member's trend, on (member, region, season)
member_trend_tree = regional_tree.map_over_datasets(lambda ds: ds.map(ev.trend))
member_trend_tree

In [ ]:
#(t): ERA5's
era5_trend_da = ev.trend(era5_regional_da)
era5_trend_da.to_pandas()

## 3. Vector

The table of regions × seasons as one vector, region by region, the seasons in `SEASONS` order inside each (`fp.flatten`; `fp.unflatten` turns it back). Entries with no data (a region with no grid points) are left out of every vector alike.

In [ ]:
#(t): ERA5's trends as one vector, every region and season
y_full_da = fp.flatten(era5_trend_da.sel(region=list(REGIONS), season=SEASONS))
y_full_da

In [ ]:
#(t): The entries with data
entry_valid_arr = y_full_da.notnull().values
y_da = y_full_da.isel(entry=entry_valid_arr)
y_arr = y_da.values
y_da.to_pandas()

In [ ]:
#(t): Every member's trends as vectors with the same entries, on (member, entry)
member_vector_tree = member_trend_tree.sel(region=list(REGIONS), season=SEASONS).map_over_datasets(fp.flatten)
member_vector_tree = member_vector_tree.isel(entry=entry_valid_arr)
member_vector_tree

In [ ]:
#(t): Check: the vector back as a table is ERA5's trends
fp.unflatten(y_da).to_pandas()

## 4. Fingerprints

For every member of every model and experiment, the same vector; their ensemble mean in each model; then each signal (`fp.SIGNAL_SETS`), its experiments' ensemble means times their coefficients, averaged over the models with equal weight:

| Set | Signals |
|---|---|
| 1 signal | ALL (historical) |
| 2 signals | ANT (historical − hist-nat), NAT |
| 3 signals | GHG, OTH (hist-aer + hist-totalO3), NAT |
| 4 signals | GHG, AER, O3, NAT |

More than one signal assumes the forcings add up (notebook 03, section 6); if four are degenerate, report three. Each fingerprint keeps some internal variability, less the more members it averages. Its **effective ensemble size** says how much: a signal $\sum_e c_e \bar{x}_e$ from $M$ models with $m_{e,j}$ members has

$$
n_{\mathrm{eff}} = \left[\frac{1}{M^2} \sum_j \sum_e \frac{c_e^2}{m_{e,j}}\right]^{-1},
$$

which TLS needs, and which is smallest for ozone (hist-totalO3 has the fewest members).

In [ ]:
%%time
#(t): Each model's ensemble-mean vector, on (model, experiment, entry)
ensemble_mean_da = datatree.reduce_to_dataset(member_vector_tree, lambda ds: ds.mean('member'))[VARIABLE]
ensemble_mean_da

In [ ]:
#(t): How many members each ensemble has, on (model, experiment)
member_count_da = datatree.reduce_to_dataset(member_vector_tree,
                                             lambda ds: xr.Dataset({'members': ds.sizes['member']}))['members']
member_count_da.to_pandas()

In [ ]:
#(t): Every model's fingerprints of every signal set, on (signal, model, entry)
model_fingerprints = {name: fp.model_fingerprints(ensemble_mean_da, signals) for name, signals in fp.SIGNAL_SETS.items()}
model_fingerprints[MAIN_SET]

In [ ]:
#(t): The multi-model fingerprints, on (signal, entry)
fingerprints = {name: prints.mean('model') for name, prints in model_fingerprints.items()}
fingerprints[MAIN_SET].to_pandas().T

In [ ]:
#(t): Their effective ensemble sizes
n_effs = {name: fp.effective_members(member_count_da, signals) for name, signals in fp.SIGNAL_SETS.items()}
pd.DataFrame({name: n_eff.to_series() for name, n_eff in n_effs.items()}).round(1)

In [ ]:
#(t): Quick look: ERA5's trends as a table of regions x seasons
fp.unflatten(y_da).to_pandas().round(2)

## 5. Noise and Its Covariance

Each sample is a member's vector minus its own ensemble's mean, scaled by $\sqrt{m/(m-1)}$ to put back the share of its noise the mean took (`fp.ensemble_anomalies`). Every model and experiment is pooled: each experiment gives its own independent realisations of internal variability. This assumes the models' variability is the real world's; notebook 02's noise tests support or undermine that, and `NOISE_INFLATION` scales it up if ERA5 is noisier.

The samples are split into two independent halves, alternating within each ensemble (`fp.pool_samples`):

- $\mathbf{C}_1$ from the first half, shrunk by Ledoit–Wolf towards a multiple of the identity (`fp.ledoit_wolf`; a few hundred samples of a few dozen entries give a noisy sample covariance, whose smallest eigenvalues inverting would turn into the largest weights). It weights the fit.
- $\mathbf{C}_2$ from the second half: the uncertainty and the residual test, so that neither uses the noise the fit was weighted with.

In [ ]:
#(t): Every member minus its ensemble's mean, scaled by sqrt(m / (m - 1))
anomaly_tree = member_vector_tree.map_over_datasets(fp.ensemble_anomalies)
anomaly_tree

In [ ]:
#(t): Pooled, on (sample, entry), with the half each sample is in
noise_da = fp.pool_samples(anomaly_tree, VARIABLE)
noise_da

In [ ]:
#(t): The two halves as arrays, (samples, entries)
noise_1_arr = noise_da.where(noise_da['half'] == 0, drop=True).values
noise_2_arr = noise_da.where(noise_da['half'] == 1, drop=True).values
noise_1_arr.shape, noise_2_arr.shape

In [ ]:
#(t): The Ledoit-Wolf covariance of each half, and how much each was shrunk
covariance_1_arr, shrinkage_1 = fp.ledoit_wolf(noise_1_arr)
covariance_2_arr, shrinkage_2 = fp.ledoit_wolf(noise_2_arr)
pd.Series({'shrinkage, half 1': shrinkage_1, 'shrinkage, half 2': shrinkage_2}).round(3)

In [ ]:
#(t): Scaled by NOISE_INFLATION
covariance_1_arr = NOISE_INFLATION * covariance_1_arr
covariance_2_arr = NOISE_INFLATION * covariance_2_arr
covariance_2_arr

In [ ]:
#(t): Quick look: C1 as correlations, and the noise standard deviation of each entry
noise_std_arr = np.sqrt(np.diag(covariance_1_arr))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
image = axes[0].imshow(covariance_1_arr / np.outer(noise_std_arr, noise_std_arr), cmap='RdBu_r', vmin=-1, vmax=1)
fig.colorbar(image, ax=axes[0])
axes[0].set_title('C1 as correlations (entries region by region)')
fp.unflatten(xr.DataArray(noise_std_arr, dims='entry', coords=y_da.coords)).plot.line(x='season', hue='region', ax=axes[1])
axes[1].set_title('Noise standard deviation of each entry (°C per decade)');

## 6. Prewhitening

$\mathbf{P} = \mathbf{C}_1^{-1/2}$, from $\mathbf{C}_1$'s eigendecomposition (`fp.inverse_sqrt`). After $\tilde{\mathbf{y}} = \mathbf{P}\mathbf{y}$ and $\tilde{\mathbf{X}} = \mathbf{P}\mathbf{X}$ the noise is uncorrelated with equal variance, and the rest is an ordinary regression.

In [ ]:
whitening_arr = fp.inverse_sqrt(covariance_1_arr)
whitening_arr

In [ ]:
#(t): Check: P C1 Pᵀ is the identity
float(np.abs(whitening_arr @ covariance_1_arr @ whitening_arr.T - np.eye(len(y_arr))).max())

In [ ]:
#(t): MAIN_SET's fingerprints as columns, (entries, signals), and their effective ensemble sizes
X_arr = fingerprints[MAIN_SET].transpose('entry', 'signal').values
n_eff_arr = n_effs[MAIN_SET].sel(signal=fingerprints[MAIN_SET]['signal']).values
X_arr.shape, n_eff_arr

In [ ]:
#(t): Quick look: the observations and the fingerprints before (left) and after (right) prewhitening
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].plot(y_arr, 'k-o', label='y')
axes[0].plot(X_arr, '-', label=list(fingerprints[MAIN_SET].signal.values))
axes[1].plot(whitening_arr @ y_arr, 'k-o')
axes[1].plot(whitening_arr @ X_arr, '-')
axes[0].set_title('y and X'); axes[1].set_title('Py and PX'); axes[0].legend(ncol=5)
for ax in axes:
    ax.set_xlabel('entry')

## 7. Fit

**OLS** (generalised least squares): $\hat{\boldsymbol\beta} = (\tilde{\mathbf{X}}^\top\tilde{\mathbf{X}})^{-1}\tilde{\mathbf{X}}^\top\tilde{\mathbf{y}} = (\mathbf{X}^\top\mathbf{C}_1^{-1}\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{C}_1^{-1}\mathbf{y}$. It treats the fingerprints as exact, so for noisy ones (ozone) $\hat\beta$ is pulled towards 0.

**TLS** allows the fingerprints noise too, $1/n_{\mathrm{eff}}$ of the observations': each prewhitened fingerprint is scaled by $\sqrt{n_{\mathrm{eff},i}}$, the right singular vector $\mathbf{v}$ of the smallest singular value of $\mathbf{Z} = [\tilde{\mathbf{X}}_s, \tilde{\mathbf{y}}]$ gives $\boldsymbol\beta_s = -\mathbf{v}_{1..k}/v_{k+1}$, and $\beta_i = \beta_{s,i}\sqrt{n_{\mathrm{eff},i}}$.

OLS first, to see the answer; TLS for the final result. First MAIN_SET, then every set.

In [ ]:
#(t): MAIN_SET by OLS
beta_ols_arr = fp.fit_ols(y_arr, X_arr, covariance_1_arr)
beta_ols_arr

In [ ]:
#(t): and by TLS
beta_tls_arr = fp.fit_tls(y_arr, X_arr, covariance_1_arr, n_eff_arr)
pd.DataFrame({'OLS': beta_ols_arr, 'TLS': beta_tls_arr}, index=fingerprints[MAIN_SET].signal.values).round(2)

In [ ]:
#(t): Every set's fingerprints as columns, and its effective ensemble sizes in the same order
X_arrs = {name: prints.transpose('entry', 'signal').values for name, prints in fingerprints.items()}
n_eff_arrs = {name: n_effs[name].sel(signal=fingerprints[name]['signal']).values for name in fingerprints}
signal_names = {name: list(prints['signal'].values) for name, prints in fingerprints.items()}

In [ ]:
#(t): Every set by both fits
betas = {(name, method): fp.fit(y_arr, X_arrs[name], covariance_1_arr, n_eff_arrs[name], method)
         for name in fp.SIGNAL_SETS for method in METHODS}
fp.fit_table({key: {'beta': beta} for key, beta in betas.items()}, signal_names)['beta'].to_dataframe().unstack('fit').round(2)

## 8. Uncertainty: Monte Carlo

Each trial adds a noise sample from the second half to the observations, $\mathbf{y}^* = \mathbf{y} + \boldsymbol\varepsilon$, and to each fingerprint another divided by $\sqrt{n_{\mathrm{eff},i}}$, $\mathbf{x}_i^* = \mathbf{x}_i + \boldsymbol\varepsilon'_i/\sqrt{n_{\mathrm{eff},i}}$ (samples drawn with replacement), and refits. The 5th to 95th percentiles of the trials' β are the 90% range. For OLS, the analytic $\mathrm{Var}(\hat{\boldsymbol\beta}) = \mathbf{A}\mathbf{C}_2\mathbf{A}^\top$, with $\mathbf{A} = (\mathbf{X}^\top\mathbf{C}_1^{-1}\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{C}_1^{-1}$, is a check (it leaves out the fingerprints' noise).

In [ ]:
%%time
#(t): The trials' β of every set and fit, (trials, signals)
trials = {(name, method): fp.monte_carlo(y_arr, X_arrs[name], covariance_1_arr, noise_2_arr, n_eff_arrs[name], method,
                                         n_trials=N_MONTE_CARLO, seed=0)
          for name in fp.SIGNAL_SETS for method in METHODS}

In [ ]:
#(t): β and its range in one Dataset on (set, fit, signal)
betas_ds = fp.fit_table({key: {'beta': betas[key], 'lower': np.percentile(trials[key], RANGE[0], axis=0),
                               'upper': np.percentile(trials[key], RANGE[1], axis=0)} for key in betas}, signal_names)
betas_ds

In [ ]:
#(t): Check: MAIN_SET's OLS range from the trials against the analytic one (±1.645 standard deviations)
analytic_std_arr = np.sqrt(np.diag(fp.ols_covariance(X_arr, covariance_1_arr, covariance_2_arr)))
pd.DataFrame({'Monte Carlo half-width': (np.percentile(trials[(MAIN_SET, 'ols')], RANGE[1], axis=0)
                                         - np.percentile(trials[(MAIN_SET, 'ols')], RANGE[0], axis=0)) / 2,
              'analytic half-width': 1.645 * analytic_std_arr}, index=signal_names[MAIN_SET]).round(3)

## 9. Residual Consistency Test

What the fit leaves over, $\mathbf{r} = \mathbf{y} - \mathbf{X}\hat{\boldsymbol\beta}$, should be internal variability. Its size, $\mathbf{r}^\top\mathbf{C}_2^{-1}\mathbf{r}$, is compared with its distribution when the fit is right: pseudo-observations $\mathbf{X}\hat{\boldsymbol\beta}$ plus a noise sample, fitted the same way, each giving its own statistic (`fp.residual_test`). In the top 5% (p < 0.05), the fit is inconsistent and its β should not be trusted. Then the entries that drive the residual are a result in themselves.

In [ ]:
%%time
#(t): The test of every set, with the final fit (METHOD)
residual_tests = {name: fp.residual_test(y_arr, X_arrs[name], betas[(name, METHOD)], covariance_1_arr,
                                         covariance_2_arr, noise_2_arr, n_eff_arrs[name], METHOD,
                                         n_trials=N_MONTE_CARLO, seed=1)
                  for name in fp.SIGNAL_SETS}
pd.DataFrame({name: {'statistic': test['statistic'], 'p': test['pvalue'], 'consistent': test['pvalue'] >= 0.05}
              for name, test in residual_tests.items()}).T

In [ ]:
#(t): Which entries drive the residual: MAIN_SET's residual over each entry's noise standard deviation
residual_arr = y_arr - X_arr @ betas[(MAIN_SET, METHOD)]
standardised_residual_da = xr.DataArray(residual_arr / np.sqrt(np.diag(covariance_2_arr)), dims='entry',
                                        coords=y_da.coords, name='standardised residual')
fp.unflatten(standardised_residual_da).to_pandas().round(2)

In [ ]:
#(t): MAIN_SET's fit by the final fit (METHOD), as a vector
fit_arr = X_arr @ betas[(MAIN_SET, METHOD)]
fit_arr

## 10. What It Says

For each set and fit: detected (the 90% range excludes 0), consistent (it includes 1), and attributable (both, and the residual test passes). Then each signal's attributable trend, $\beta_i \mathbf{x}_i$ with the range of $\beta_i$, region by region and season by season, next to the observed trend.

In [ ]:
#(t): Detected and consistent, every set, fit and signal
verdict_ds = xr.Dataset({
    'detected': (betas_ds['lower'] > 0) | (betas_ds['upper'] < 0),
    'consistent': (betas_ds['lower'] <= 1) & (betas_ds['upper'] >= 1),
}).where(betas_ds['beta'].notnull())
verdict_ds.to_dataframe().dropna().unstack('fit')

In [ ]:
#(t): MAIN_SET's β and its range by the final fit, on (signal)
main_beta_ds = betas_ds.sel(set=MAIN_SET, fit=METHOD).dropna('signal')
main_beta_ds.to_dataframe().round(2)

In [ ]:
#(t): The attributable trends, β times the fingerprint, with the range of β (whichever end gives the lower trend)
main_fingerprint_da = fingerprints[MAIN_SET]
attributable_ds = xr.Dataset({
    'trend': main_beta_ds['beta'] * main_fingerprint_da,
    'lower': np.minimum(main_beta_ds['lower'] * main_fingerprint_da, main_beta_ds['upper'] * main_fingerprint_da),
    'upper': np.maximum(main_beta_ds['lower'] * main_fingerprint_da, main_beta_ds['upper'] * main_fingerprint_da),
})
attributable_ds = fp.unflatten(attributable_ds)
attributable_ds

In [ ]:
#(t): Quick look: MAIN_SET's attributable trends of the first region, every signal and season
attributable_ds['trend'].isel(region=0).to_pandas().round(2)

## 11. Collinearity

Greenhouse gases and ozone both trend over 1979–2000, so in time alone they look alike; the seasons are what separate them (ozone is mostly summer). Two checks: the correlation of the fingerprints across the entries, and the correlation of the signals' β across the Monte Carlo trials. Above about 0.8 the two cannot be told apart: report their combined effect (the 3-signal set) rather than each.

In [ ]:
#(t): The fingerprints' correlation over the entries, MAIN_SET
fingerprint_correlation_da = xr.corr(main_fingerprint_da, main_fingerprint_da.rename(signal='signal_2'), dim='entry')
fingerprint_correlation_da.to_pandas().round(2)

In [ ]:
#(t): The β's correlation across the trials, MAIN_SET by the final fit
beta_correlation_da = xr.DataArray(np.corrcoef(trials[(MAIN_SET, METHOD)].T), dims=('signal', 'signal_2'),
                                   coords={'signal': signal_names[MAIN_SET], 'signal_2': signal_names[MAIN_SET]})
beta_correlation_da.to_pandas().round(2)

## 12. Robustness

- **Each model's own fingerprints** (`fp.fit_each`), with the effective ensemble size of that model alone: the spread of β between models.
- **Leaving each model out** of the multi-model fingerprints.

Still to add (planning notes, 10.9): other periods (1979–2014 vs 1979–2020 needs a longer LESFMIP record), other regions, ERA5 against station records (READER), and the trend against a late-minus-early difference.

In [ ]:
#(t): Each model's own effective ensemble sizes, on (signal, model)
model_n_eff_da = fp.model_effective_members(member_count_da, fp.SIGNAL_SETS[MAIN_SET])
model_n_eff_da.to_pandas()

In [ ]:
#(t): β with each model's own fingerprints, on (model, signal)
each_model_beta_da = fp.fit_each(y_da, model_fingerprints[MAIN_SET], model_n_eff_da, covariance_1_arr, METHOD)
each_model_beta_da.to_pandas().round(2)

In [ ]:
#(t): The multi-model fingerprints without each model in turn ('model' is the one left out)
n_models_da = model_fingerprints[MAIN_SET].notnull().all('entry').sum('model')
left_out_prints_da = (model_fingerprints[MAIN_SET].sum('model') - model_fingerprints[MAIN_SET]) / (n_models_da - 1)
left_out_prints_da

In [ ]:
#(t): and their effective ensemble sizes: (M - 1)² over the other models' sum of 1 / n_eff
inverse_n_eff_da = 1 / model_n_eff_da
left_out_n_eff_da = (n_models_da - 1) ** 2 / (inverse_n_eff_da.sum('model') - inverse_n_eff_da)
left_out_n_eff_da

In [ ]:
#(t): β leaving each model out, on (model, signal)
left_out_beta_da = fp.fit_each(y_da, left_out_prints_da, left_out_n_eff_da, covariance_1_arr, METHOD)
left_out_beta_da.to_pandas().round(2)

## Saved Results

Everything notebook 08b draws: the scaling factors, the attributable trends, and, for its figures, the tables of the observations, the fingerprints, the fit and the residual, the residual tests, the collinearity and the robustness.

In [ ]:
#(t): β and its range, on (set, fit, signal)
betas_path = RESULTS_DIR / 'fingerprinting_betas.zarr'
betas_ds.to_zarr(betas_path, mode='w')
betas_path

In [ ]:
#(t): The attributable trends, on (signal, region, season)
attributable_path = RESULTS_DIR / 'fingerprinting_attributable.zarr'
attributable_ds.to_zarr(attributable_path, mode='w')
attributable_path

In [ ]:
#(t): The tables of regions x seasons: ERA5's trends, MAIN_SET's fingerprints, and its fit and residual by METHOD
fingerprinting_tables_ds = xr.Dataset({
    'era5': fp.unflatten(y_da),
    'fingerprint': fp.unflatten(fingerprints[MAIN_SET]),
    'fit': fp.unflatten(y_da.copy(data=fit_arr)),
    'residual': fp.unflatten(y_da.copy(data=residual_arr)),
})
fingerprinting_tables_ds

In [ ]:
tables_path = RESULTS_DIR / 'fingerprinting_tables.zarr'
fingerprinting_tables_ds.to_zarr(tables_path, mode='w')
tables_path

In [ ]:
#(t): The residual test of every set (METHOD): the observed statistic, its null distribution and the p-value
residual_tests_ds = xr.Dataset({
    'statistic': ('set', [residual_tests[name]['statistic'] for name in residual_tests]),
    'null': (('set', 'trial'), np.stack([residual_tests[name]['null'] for name in residual_tests])),
    'pvalue': ('set', [residual_tests[name]['pvalue'] for name in residual_tests]),
}, coords={'set': list(residual_tests)})
residual_tests_ds

In [ ]:
residual_tests_path = RESULTS_DIR / 'fingerprinting_residual_tests.zarr'
residual_tests_ds.to_zarr(residual_tests_path, mode='w')
residual_tests_path

In [ ]:
#(t): The collinearity of MAIN_SET: the fingerprints' correlation over the entries, and the β's across the trials
collinearity_ds = xr.Dataset({'fingerprints': fingerprint_correlation_da, 'beta': beta_correlation_da})
collinearity_ds

In [ ]:
collinearity_path = RESULTS_DIR / 'fingerprinting_collinearity.zarr'
collinearity_ds.to_zarr(collinearity_path, mode='w')
collinearity_path

In [ ]:
#(t): β with each model's own fingerprints, and leaving each model out, on (model, signal)
robustness_ds = xr.Dataset({'each_model': each_model_beta_da, 'left_out': left_out_beta_da})
robustness_ds

In [ ]:
robustness_path = RESULTS_DIR / 'fingerprinting_robustness.zarr'
robustness_ds.to_zarr(robustness_path, mode='w')
robustness_path